# Lab 4 — Build the Audience, Then Buy It
Name: Alexander Zavadskyi | Group: KM-41

## Setup

In [1]:
import importlib.util, subprocess, sys
required = ['kagglehub', 'pandas', 'numpy', 'pyarrow', 'matplotlib', 'scipy']
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
DATA = kagglehub.dataset_download('shkarupylomaxim/adtech-lab4-tv-audiences/versions/2')
for name in ['programs', 'viewings', 'daily_weights', 'viewer_attributes', 'attributes',
             'rate_card', 'acr_sessions', 'hh_attributes', 'reference_ratings', 'universe']:
    globals()[name] = pd.read_parquet(f'{DATA}/{name}.parquet')
survey = pd.read_csv(f'{DATA}/purchase_survey.txt', sep='\t', dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f'{DATA}/purchase_survey_layout.txt', sep='\t', dtype=str, keep_default_na=False)
dates = pd.date_range('2024-10-01', '2024-10-31')
for frame in [programs, daily_weights]:
    frame['broadcast_date'] = pd.to_datetime(frame.broadcast_date)
dw = daily_weights.loc[daily_weights.resp_type.isin(['R', 'L']) & daily_weights.broadcast_date.isin(dates)].copy()
assert not dw.duplicated(['respondent_id', 'broadcast_date']).any()
age = viewer_attributes.loc[viewer_attributes.attr_id.eq(300)].set_index('respondent_id').value
people = set(viewer_attributes.loc[viewer_attributes.attr_id.isin([101, 102]), 'respondent_id'])
adults = set(age.index[age.ge(18)]) & people
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print('Dataset version: 2')

C:\Users\leks\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset version: 2


## Task 1.0 — Clean the survey file

In [2]:
display(survey.head(10))
print('Raw VALUE tokens:', survey.VALUE.value_counts(dropna=False).to_dict())
layout = survey_layout.loc[survey_layout.RECTYPE.eq('B')].copy()
s = survey.copy()
audit = []
def note(kind, mask, action):
    audit.append({'kind of bad record': kind, 'rows': int(mask.sum()), 'action': action})
note('RESPID whitespace', s.RESPID.ne(s.RESPID.str.strip()), 'Strip whitespace')
note('RESPID decimal suffix', s.RESPID.str.strip().str.endswith('.0'), 'Remove integer .0 suffix')
note('RESPID leading zero', s.RESPID.str.strip().str.match(r'^0[0-9]+'), 'Parse as integer')
note('KEYWORD case / whitespace', s.KEYWORD.ne(s.KEYWORD.str.strip().str.upper()), 'Strip and uppercase')
rid = s.RESPID.str.strip().str.replace(r'\.0$', '', regex=True)
s['respondent_id'] = pd.to_numeric(rid.where(rid.str.fullmatch(r'[0-9]+')), errors='coerce').astype('Int64')
s['KEYWORD'] = s.KEYWORD.str.strip().str.upper()
note('Embedded header / footer', s.respondent_id.isna(), 'Drop nonnumeric ids')
note('Not an adult R/L id', s.respondent_id.notna() & ~s.respondent_id.isin(adults), 'Drop households, children, guests and unknown ids')
note('Unknown answer code', ~s.KEYWORD.isin(layout.KEYWORD) & s.respondent_id.notna(), 'Drop codes outside answer codebook')
yes = s.VALUE.str.strip().str.lower().isin(['1', '1.0', 'y', 'yes', 'true'])
note('Alternative yes token', yes & s.VALUE.ne('1'), 'Normalize to yes')
note('No / blank value', ~yes & s.respondent_id.notna(), 'Drop; export should contain yes answers only')
clean = s.loc[yes & s.respondent_id.isin(adults) & s.KEYWORD.isin(layout.KEYWORD), ['respondent_id', 'KEYWORD']].copy()
note('Duplicate person / keyword', clean.duplicated(), 'Keep one normalized yes answer')
clean = clean.drop_duplicates()
pair_codes = {'X710101': 1, 'X710102': 2, 'X710103': 3}
pair_rows = clean.loc[clean.KEYWORD.isin(pair_codes)].copy()
pair_rows['pairs'] = pair_rows.KEYWORD.map(pair_codes)
conflicts = pair_rows.groupby('respondent_id').size().gt(1)
max_pairs = pair_rows.groupby('respondent_id').pairs.max()
discard = clean.KEYWORD.isin(pair_codes) & clean.KEYWORD.map(pair_codes).lt(clean.respondent_id.map(max_pairs))
note('Lower conflicting pair count', discard, 'Keep higher pair category per person')
clean = clean.loc[~discard].copy()
purchase = pd.DataFrame(index=pd.Index(sorted(adults), name='respondent_id'))
purchase['pairs'] = max_pairs.reindex(purchase.index, fill_value=0).astype(int)
for brand, keyword in [('STRIDE', 'X710201'), ('KESTREL', 'X710202'), ('EASYWALK', 'X710203')]:
    purchase[brand] = purchase.index.isin(clean.loc[clean.KEYWORD.eq(keyword), 'respondent_id'])
purchase['buyer'] = purchase.index.isin(clean.respondent_id)
missing_any = set(purchase.index[purchase.buyer]) - set(clean.loc[clean.KEYWORD.eq('X710100'), 'respondent_id'])
print('Buyers missing Any (inferred):', len(missing_any))
# Inference restores the Any answer; all brand categories, including OTHER / DONT KNOW, imply buying.
clean = pd.concat([clean, pd.DataFrame({'respondent_id': sorted(missing_any), 'KEYWORD': 'X710100'})], ignore_index=True)
check = layout[['KEYWORD', 'Text', 'SAMPLE']].copy()
check['your count'] = check.KEYWORD.map(clean.groupby('KEYWORD').size()).fillna(0).astype(int)
check['SAMPLE'] = check.SAMPLE.astype(int)
assert check['your count'].eq(check.SAMPLE).all()
assert len(purchase) == len(adults) and purchase.index.is_unique
assert purchase.loc[~purchase.buyer, 'pairs'].eq(0).all()
print('Survey data rows:', len(survey))
print('Clean yes answers:', len(clean))
print('Adult panel members:', len(purchase))
print('Non-buyers:', int((~purchase.buyer).sum()))
print('People with two pair counts:', int(conflicts.sum()))
display(pd.DataFrame(audit))
display(check[['KEYWORD', 'your count', 'SAMPLE']])
display(purchase.assign(pairs=purchase.pairs.map({0: '0', 1: '1', 2: '2', 3: '3+'})).head(10))
# Explicit naive baseline: exact keyword and VALUE == "1", distinct raw text ids,
# Any is treated as the only buyer flag, and denominator is the known adult panel.
raw_kestrel = survey.loc[survey.KEYWORD.eq('X710202') & survey.VALUE.eq('1'), 'RESPID'].nunique()
raw_buyers = survey.loc[survey.KEYWORD.eq('X710100') & survey.VALUE.eq('1'), 'RESPID'].nunique()
print('Naive raw Kestrel count:', raw_kestrel)
print('Clean Kestrel count:', int(purchase.KESTREL.sum()))
print('Naive raw buyer share:', f'{raw_buyers / len(adults):.3f}')
print('Clean buyer share:', f'{purchase.buyer.mean():.3f}')
print('Raw Kestrel count error:', raw_kestrel - int(purchase.KESTREL.sum()))
print('Raw buyer share error (percentage points):', f'{100 * (raw_buyers / len(adults) - purchase.buyer.mean()):.1f}')

,RESPID,KEYWORD,VALUE
0,10415501,X710298,1
1,10072201,x710100,1
2,10302802,X710299,1
3,10070002,X710100,1
4,10130702,X710203,1
5,10243701,X710100,1
6,10342601,X710103,0
7,10399201,X710298,1
8,10364102,X710298,1
9,10200102,X710102,1


Raw VALUE tokens: {'1': 16410, '1.0': 262, 'y': 246, 'TRUE': 241, '0': 237, 'Y': 228, 'yes': 217, '0.0': 129, 'N': 127, 'no': 126, '': 124, 'VALUE': 2}
Buyers missing Any (inferred): 238
Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,kind of bad record,rows,action
0,RESPID whitespace,573,Strip whitespace
1,RESPID decimal suffix,533,Remove integer .0 suffix
2,RESPID leading zero,349,Parse as integer
3,KEYWORD case / whitespace,1104,Strip and uppercase
4,Embedded header / footer,3,Drop nonnumeric ids
5,Not an adult R/L id,423,"Drop households, children, guests and unknown ids"
6,Unknown answer code,738,Drop codes outside answer codebook
7,Alternative yes token,1194,Normalize to yes
8,No / blank value,742,Drop; export should contain yes answers only
9,Duplicate person / keyword,933,Keep one normalized yes answer


,KEYWORD,your count,SAMPLE
2,X710100,4779,4779
3,X710101,1810,1810
4,X710102,1308,1308
5,X710103,1661,1661
7,X710201,1563,1563
8,X710202,354,354
9,X710203,1050,1050
10,X710298,2739,2739
11,X710299,371,371


,pairs,STRIDE,KESTREL,EASYWALK,buyer
respondent_id,,,,,
10000101,3+,False,False,False,True
10000102,2,False,False,True,True
10000201,1,True,False,False,True
10000301,1,False,False,True,True
10000302,0,False,False,False,False
10000401,1,False,False,False,True
10000501,0,False,False,False,False
10000502,1,True,False,False,True
10000601,3+,True,False,False,True


Naive raw Kestrel count: 366
Clean Kestrel count: 354
Naive raw buyer share: 0.503
Clean buyer share: 0.597
Raw Kestrel count error: 12
Raw buyer share error (percentage points): -9.3


**Answer:** The audit reports affected rows at each cleaning stage; categories can overlap and are not additive. IDs need whitespace, leading-zero and `.0` normalization, codes need case and whitespace normalization, and yes tokens need interpretation. Headers, the trailer, nonadult/nonpanel IDs, unknown codes and no/blank answers are excluded; duplicate answers collapse, 113 conflicting counts retain the higher value, and any valid purchase answer implies a buyer (restoring 238 missing Any answers). The codebook check covers all nine answers, including inferred Any flags. The explicit raw-file baseline gives 366 Kestrel buyers instead of 354, an overcount of 12 (3.4%), and a buyer share of 0.503 instead of 0.597, an understatement of 9.3 percentage points. This baseline uses distinct unnormalized IDs with exact code and value `1`; other naive interpretations of the dirty file would produce different errors.

## Task 1.1 — Four definitions

In [3]:
audiences = {
    'Adults 18+': adults,
    'Shoe buyers': set(purchase.index[purchase.buyer]),
    'Heavy buyers': set(purchase.index[purchase.pairs.eq(3)]),
    'Kestrel buyers': set(purchase.index[purchase.KESTREL]),
    'Adults 25-54': adults & set(age.index[age.between(25, 54)]),
    'P2+': people,
}
def daily_size(ids):
    return dw.loc[dw.respondent_id.isin(ids)].groupby('broadcast_date').weight.sum().reindex(dates, fill_value=0)
universes = pd.DataFrame({name: daily_size(ids) for name, ids in audiences.items()})
sizes = universes.mean()
for label, name in [('Adults 18+ (M):', 'Adults 18+'), ('Shoe buyers (M):', 'Shoe buyers'),
                    ('Heavy buyers, 3+ pairs (M):', 'Heavy buyers'), ('Kestrel buyers (M):', 'Kestrel buyers'),
                    ('Adults 25-54 (M):', 'Adults 25-54')]:
    print(label, f'{sizes[name] / 1e6:.1f}')
print('Kestrel buyers in panel (n):', len(audiences['Kestrel buyers']))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [4]:
groups = {label: set(age.index[mask]) & adults for label, mask in {
    '18-34': age.between(18, 34), '35-54': age.between(35, 54), '55+': age.ge(55)}.items()}
for label, attr_id in [('Men', 201), ('Women', 202)]:
    groups[label] = set(viewer_attributes.loc[viewer_attributes.attr_id.eq(attr_id), 'respondent_id']) & adults
profile_rows = []
for name, ids in list(audiences.items())[:5]:
    for group, members in groups.items():
        share = daily_size(ids & members).mean() / sizes[name]
        adult_share = daily_size(adults & members).mean() / sizes['Adults 18+']
        profile_rows.append((name, group, share, share / adult_share * 100))
profiles = pd.DataFrame(profile_rows, columns=['audience', 'group', 'share', 'index vs adults'])
display(profiles.style.format({'share': '{:.3f}', 'index vs adults': '{:.1f}'}))
print('Buyer share of adult universe:', f'{sizes["Shoe buyers"] / sizes["Adults 18+"]:.3f}')

,audience,group,share,index vs adults
0,Adults 18+,18-34,0.288,100.0
1,Adults 18+,35-54,0.319,100.0
2,Adults 18+,55+,0.393,100.0
3,Adults 18+,Men,0.488,100.0
4,Adults 18+,Women,0.512,100.0
5,Shoe buyers,18-34,0.281,97.5
6,Shoe buyers,35-54,0.336,105.3
7,Shoe buyers,55+,0.383,97.5
8,Shoe buyers,Men,0.485,99.3
9,Shoe buyers,Women,0.515,100.7


Buyer share of adult universe: 0.586


**Answer:** Use heavy buyers (3+ pairs) as the planning target: it matches Kestrel's brief and includes potential customers of competing brands. Kestrel buyers are a smaller retention audience; adults 25–54 are only a demographic proxy and exclude relevant buyers. Athletic-shoe buying spans all adult demographic groups and represents 58.6% of the adult universe, so its profile resembles adults more than a narrowly defined heavy-buyer segment does; this does not mean nearly every adult buys shoes. The handout says six groups but lists five: the table shows all five specified groups (three age bands and two sexes). A fused purchase attribute cannot establish that a program itself attracts extra shoe buyers beyond its demographic composition, or prove that its viewers will respond to Kestrel advertising.

## Task 2.1 — Programs by audience

In [5]:
prime = programs.loc[programs.daypart.eq('Prime') & programs.broadcast_date.isin(dates)].copy()
sessions = {net: g for net, g in viewings.loc[viewings.resp_type.isin(['R', 'L'])].groupby('network_code')}
day_weights = {day: g.set_index('respondent_id').weight for day, g in dw.groupby('broadcast_date')}
# Average-minute ratings use all overlapping minutes, not the ACR one-minute reach rule.
rating_rows = []
for t in prime.itertuples(index=False):
    g = sessions[t.network_code]
    g = g.loc[g.start_ts.lt(t.end_ts) & g.end_ts.gt(t.start_ts)]
    minutes = (g.end_ts.clip(upper=t.end_ts) - g.start_ts.clip(lower=t.start_ts)).dt.total_seconds() / 60
    viewed = minutes.groupby(g.respondent_id).sum()
    weights = day_weights[t.broadcast_date].reindex(viewed.index)
    assert weights.notna().all()
    assert viewed.le(t.duration_min + 1e-8).all(), 'Overlapping person sessions must not double count'
    row = {'telecast_id': t.telecast_id}
    for name, ids in audiences.items():
        mask = viewed.index.isin(ids)
        imp = (viewed[mask] * weights[mask]).sum() / t.duration_min
        row[name + ' viewers'] = imp
        row[name + ' rating'] = imp / universes.loc[t.broadcast_date, name] * 100
        row[name + ' panel viewers'] = int(mask.sum())
    rating_rows.append(row)
ratings = prime.merge(pd.DataFrame(rating_rows), on='telecast_id', validate='one_to_one')
def price(t):
    rows = rate_card.loc[rate_card.network_code.eq(t.network_code)]
    special = rows.loc[rows.program_name.eq(t.program_name)]
    if len(special):
        assert len(special) == 1
        return special.cost_30s_usd.iloc[0]
    minute = (t.start_ts - t.broadcast_date).total_seconds() / 60
    def clock(s):
        h, m = map(int, s.split(':'))
        return 60*h + m
    choices = [r.cost_30s_usd for r in rows.loc[rows.program_name.isna()].itertuples()
               if t.broadcast_date.dayofweek in r.weekdays and clock(r.start_time) <= minute < clock(r.end_time)]
    assert len(choices) == 1, (t.telecast_id, choices)
    return choices[0]
ratings['cost per :30'] = [price(t) for t in ratings.itertuples(index=False)]
choices = [('NBC', 'NFL Football'), ('ABC', 'NFL Football'), ('ABC', 'Dancing with the Stars'),
           ('CBS', 'NCIS'), ('FNC', 'Hannity'), ('HGTV', 'Unsellable Houses'),
           ('FOOD', 'Halloween Wars'), ('FX', 'Grotesquerie')]
rows = []
for net, program in choices:
    g = ratings.loc[ratings.network_code.eq(net) & ratings.program_name.eq(program)]
    assert len(g) > 0
    m = g.select_dtypes(include='number').mean()
    rows.append({'program': f'{program} ({net})', 'telecasts': len(g),
        'adults rating': m['Adults 18+ rating'], 'target rating': m['Heavy buyers rating'],
        'target viewers (000)': m['Heavy buyers viewers']/1000,
        'composition': m['Heavy buyers viewers']/m['Adults 18+ viewers'],
        'index': m['Heavy buyers rating']/m['Adults 18+ rating']*100,
        'cost per :30': m['cost per :30'],
        'adults CPM': m['cost per :30']/m['Adults 18+ viewers']*1000,
        'target CPM': m['cost per :30']/m['Heavy buyers viewers']*1000,
        'Kestrel index': m['Kestrel buyers rating']/m['Adults 18+ rating']*100,
        'heavy panel viewers (mean)': m['Heavy buyers panel viewers'],
        'Kestrel panel viewers (mean)': m['Kestrel buyers panel viewers'],
        'adult viewers': m['Adults 18+ viewers']})
program_table = pd.DataFrame(rows).set_index('program')
display(program_table.iloc[:, :9].style.format({
    'telecasts': '{:.0f}', 'adults rating': '{:.2f}', 'target rating': '{:.2f}',
    'target viewers (000)': '{:.1f}', 'composition': '{:.3f}', 'index': '{:.1f}',
    'cost per :30': '{:.0f}', 'adults CPM': '{:.0f}', 'target CPM': '{:.0f}'}))
print('Target universe (M):', f'{sizes["Heavy buyers"]/1e6:.1f}')

,telecasts,adults rating,target rating,target viewers (000),composition,index,cost per :30,adults CPM,target CPM
program,,,,,,,,,
NFL Football (NBC),4,5.82,5.79,2865.0,0.195,99.3,487200,33,170
NFL Football (ABC),3,8.93,8.36,4149.1,0.184,93.6,723600,32,174
Dancing with the Stars (ABC),9,3.25,3.14,1564.2,0.191,96.8,158600,19,101
NCIS (CBS),3,2.21,2.43,1206.3,0.216,110.1,178800,32,148
Hannity (FNC),22,1.09,0.84,416.9,0.152,77.1,39900,15,96
Unsellable Houses (HGTV),13,0.50,0.64,317.2,0.249,126.8,12900,10,41
Halloween Wars (FOOD),9,0.28,0.33,161.9,0.228,116.2,9700,14,60
Grotesquerie (FX),25,0.26,0.25,124.2,0.188,95.9,7592,11,61


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [6]:
print('Highest index (program):', program_table['index'].idxmax())
print('Most target viewers (program):', program_table['target viewers (000)'].idxmax())
print('Lowest target CPM (program):', program_table['target CPM'].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): NFL Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** Unsellable Houses (HGTV) has both the highest index (126.8) and lowest target CPM ($41), while NFL Football (ABC) delivers the most heavy-buyer viewers: the leaders are not all the same program. Index measures relative affinity, volume measures the number of target viewers, and CPM measures their price. A high index can belong to a small audience and still buy few target impressions. I would compare target CPM and available volume first, then use index to explain fit and composition to quantify spill, rather than buying only the largest index. None of these averages measures incremental reach or sales lift.

## Task 2.3 — Kestrel buyers

In [7]:
display(program_table[['index', 'Kestrel index', 'heavy panel viewers (mean)', 'Kestrel panel viewers (mean)']]
        .rename(columns={'index': 'heavy-buyer index'}).round(1))

,heavy-buyer index,Kestrel index,heavy panel viewers (mean),Kestrel panel viewers (mean)
program,,,,
NFL Football (NBC),99.300,98.500,329.000,75.200
NFL Football (ABC),93.600,85.900,292.700,69.300
Dancing with the Stars (ABC),96.800,82.700,122.600,23.400
NCIS (CBS),110.100,56.500,86.300,11.000
Hannity (FNC),77.100,42.600,15.900,2.100
Unsellable Houses (HGTV),126.800,76.900,25.400,4.000
Halloween Wars (FOOD),116.200,95.200,13.700,2.800
Grotesquerie (FX),95.900,129.600,11.400,3.100


**Answer:** Only a small number of Kestrel buyers contribute to an individual telecast; a few viewing decisions and their weights can swing the index. The heavy-buyer index pools a substantially larger panel group and is the more stable planning signal, although it still inherits sampling error and fusion assumptions. Check the unweighted contributing counts above, household-cluster bootstrap confidence intervals, stability across weeks, and replication in an independent purchase-linked panel before relying on a brand-specific ranking.

## Task 3.1 — Weights

In [8]:
# Link using all TVs before editing, and reject any TV whose IP is not stable.
tv_ips = acr_sessions[['tv_id', 'hashed_ip']].drop_duplicates()
assert not tv_ips.tv_id.duplicated().any()
unique_hh = hh_attributes.loc[~hh_attributes.hashed_ip.duplicated(keep=False)].copy()
matched = tv_ips.merge(unique_hh, on='hashed_ip', validate='many_to_one')
assert matched.tv_id.is_unique
matched['head_age'] = np.where(matched.head_age_band.eq('65+'), '65+', 'Under 65')
a = acr_sessions.loc[(acr_sessions.end_utc - acr_sessions.start_utc).dt.total_seconds().ge(60)].copy()
for source, dest in [('start_utc', 'start_ts'), ('end_utc', 'end_ts')]:
    a[dest] = a[source].dt.tz_convert('America/New_York').dt.tz_localize(None)
# Split at 06:00 so sessions crossing a broadcast boundary count on both days.
a['broadcast_date'] = (a.start_ts - pd.Timedelta(hours=6)).dt.normalize()
crosses = a.end_ts.gt(a.broadcast_date + pd.Timedelta(days=1, hours=6))
tail = a.loc[crosses].copy()
tail['start_ts'] = tail.broadcast_date + pd.Timedelta(days=1, hours=6)
tail['broadcast_date'] += pd.Timedelta(days=1)
a.loc[crosses, 'end_ts'] = a.loc[crosses, 'broadcast_date'] + pd.Timedelta(days=1, hours=6)
a = pd.concat([a, tail], ignore_index=True)
acr_days = pd.date_range('2024-10-14', '2024-10-27')
a = a.loc[a.broadcast_date.isin(acr_days)]
in_tab = a[['tv_id', 'broadcast_date']].drop_duplicates().merge(
    matched[['tv_id', 'head_age', 'income_band']], on='tv_id', validate='many_to_one')
cell_n = in_tab.groupby(['broadcast_date', 'head_age', 'income_band']).size().rename('cell_n').reset_index()
assert len(cell_n) == len(acr_days)*len(universe), 'Every target cell must have support each day'
in_tab = in_tab.merge(cell_n, on=['broadcast_date', 'head_age', 'income_band'], validate='many_to_one')
in_tab = in_tab.merge(universe, on=['head_age', 'income_band'], validate='many_to_one')
in_tab['cell_weight'] = in_tab.tv_households / in_tab.cell_n
weight_sums = in_tab.groupby('broadcast_date').cell_weight.sum()
assert np.allclose(weight_sums, universe.tv_households.sum())
# Compare income-only weighting to demonstrate its limited correction.
income_targets = universe.groupby('income_band').tv_households.sum()
income_n = in_tab.groupby(['broadcast_date', 'income_band']).tv_id.transform('size')
in_tab['income_weight'] = in_tab.income_band.map(income_targets) / income_n
live = a.loc[a.session_type.eq('live') & a.tv_id.isin(matched.tv_id)]
acr_results = []
for t in programs.merge(reference_ratings, on='telecast_id', validate='one_to_one').itertuples(index=False):
    g = live.loc[live.broadcast_date.eq(t.broadcast_date) & live.network_code.eq(t.network_code)
                 & live.start_ts.lt(t.end_ts) & live.end_ts.gt(t.start_ts)]
    overlap = (g.end_ts.clip(upper=t.end_ts) - g.start_ts.clip(lower=t.start_ts)).dt.total_seconds()/60
    watched = set(overlap.groupby(g.tv_id).sum().loc[lambda x: x.ge(1)].index)
    tab = in_tab.loc[in_tab.broadcast_date.eq(t.broadcast_date)]
    flag = tab.tv_id.isin(watched)
    assert watched <= set(tab.tv_id)
    acr_results.append({'telecast': f'{t.program_name} ({t.network_code})', 'date': t.broadcast_date.strftime('%Y-%m-%d'),
        'unweighted': flag.mean()*100, 'weighted': tab.loc[flag, 'cell_weight'].sum()/tab.cell_weight.sum()*100,
        'income only': tab.loc[flag, 'income_weight'].sum()/tab.income_weight.sum()*100, 'reference': t.hh_rating})
acr_table = pd.DataFrame(acr_results)
gap_raw = (acr_table.unweighted-acr_table.reference).abs().mean()
gap_weighted = (acr_table.weighted-acr_table.reference).abs().mean()
print('TVs matched to one household:', len(matched))
print('Weights sum per day (M):', f'{weight_sums.min()/1e6:.1f}', 'to', f'{weight_sums.max()/1e6:.1f}')
display(acr_table.round(2))
print('Mean gap unweighted (points):', f'{gap_raw:.2f}')
print('Mean gap weighted (points):', f'{gap_weighted:.2f}')
print('Share of gap closed:', f'{1-gap_weighted/gap_raw:.3f}')
acr_table['change'] = acr_table.weighted-acr_table.unweighted
display(acr_table[['telecast', 'change']].round(2))
print('Mean absolute income-only movement (points):', f'{(acr_table["income only"]-acr_table.unweighted).abs().mean():.2f}')

TVs matched to one household: 19607
Weights sum per day (M): 125.8 to 125.8


,telecast,date,unweighted,weighted,income only,reference
0,The Real Housewives of New York City (BRAVO),2024-10-15,0.710,0.640,0.680,1.230
1,NCIS (CBS),2024-10-14,5.500,5.800,5.450,10.560
2,Hannity (FNC),2024-10-16,1.040,1.350,1.080,2.300
3,Halloween Baking Championship (FOOD),2024-10-14,0.950,0.900,0.910,1.630
4,MLB Postseason (FOX),2024-10-25,17.030,17.760,16.850,25.860
5,Grotesquerie (FX),2024-10-16,2.170,2.110,2.110,3.530
6,Unsellable Houses (HGTV),2024-10-16,2.120,2.080,2.060,3.690
7,NFL Football (NBC),2024-10-27,22.620,23.750,22.440,33.140


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


,telecast,change
0,The Real Housewives of New York City (BRAVO),-0.070
1,NCIS (CBS),0.290
2,Hannity (FNC),0.310
3,Halloween Baking Championship (FOOD),-0.050
4,MLB Postseason (FOX),0.720
5,Grotesquerie (FX),-0.060
6,Unsellable Houses (HGTV),-0.040
7,NFL Football (NBC),1.130


Mean absolute income-only movement (points): 0.08


## Task 3.2 — What weighting cannot fix

**Answer:** The change column shows the direction and size of each correction. Age weighting restores the influence of older homes, which particularly helps older-skewing CBS and Fox News; younger-skewing programs can move less or in the opposite direction. Income-only weighting barely changes ratings because income affects opt-in but adds little viewing differentiation here once age is considered. Even exact cell totals cannot recover viewing on other sets in the same household, identify individual viewers, turn unknown fingerprints into network viewing, or observe apps and unmatched/opted-out homes. Selection within cells, household attribute errors and feed outages remain; weighting a single-device footprint does not make it a complete household currency. Gaps here are mean absolute differences in rating points, so improvements and deteriorations cannot cancel.

## Task 4.1 — October predicts October

In [9]:
big = ['CBS', 'NBC', 'ABC', 'FOX', 'FNC']
history = ratings.loc[ratings.network_code.isin(big) & ratings.broadcast_date.between('2024-10-01', '2024-10-21')]
future = ratings.loc[ratings.network_code.isin(big) & ratings.broadcast_date.between('2024-10-22', '2024-10-28')]
keys = ['network_code', 'program_name']
hist_means = history.groupby(keys)[['P2+ rating', 'Heavy buyers rating']].mean().add_suffix(' forecast')
forecast = future.merge(hist_means, on=keys, how='inner', validate='many_to_one')
for name in ['P2+', 'Heavy buyers']:
    actual = forecast[name+' rating']
    forecast[name+' APE'] = (actual - forecast[name+' rating forecast']).abs() / actual.replace(0, np.nan)
p2_mape = forecast['P2+ APE'].mean()*100
heavy_mape = forecast['Heavy buyers APE'].mean()*100
worst = forecast.sort_values('P2+ APE', ascending=False).iloc[0]
print('Telecasts forecast (n):', len(forecast))
print('MAPE, P2+ (%):', f'{p2_mape:.1f}')
print('MAPE, heavy buyers (%):', f'{heavy_mape:.1f}')
print('Biggest miss, P2+ (program, date):', f'{worst.program_name} ({worst.network_code})', worst.broadcast_date.strftime('%Y-%m-%d'))
print('Nonzero actuals, P2+:', int(forecast['P2+ APE'].notna().sum()))
print('Nonzero actuals, heavy buyers:', int(forecast['Heavy buyers APE'].notna().sum()))
assert forecast.broadcast_date.min() > history.broadcast_date.max()

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX) 2024-10-27
Nonzero actuals, P2+: 101
Nonzero actuals, heavy buyers: 101


## Task 4.2 — Why it missed

In [10]:
misses = forecast.nlargest(5, 'P2+ APE').copy()
misses['program'] = misses.program_name + ' (' + misses.network_code + ')'
misses['date'] = misses.broadcast_date.dt.strftime('%Y-%m-%d')
display(misses[['program', 'date', 'P2+ rating forecast', 'P2+ rating', 'P2+ APE', 'Heavy buyers APE']]
        .rename(columns={'P2+ rating forecast': 'forecast', 'P2+ rating': 'actual'}).round(3))
# Inspect the actual schedule and preceding program, not just the error ranking.
context = []
for t in misses.itertuples(index=False):
    schedule = programs.loc[programs.network_code.eq(t.network_code) & programs.broadcast_date.eq(t.broadcast_date)].sort_values('start_ts')
    prior = schedule.loc[schedule.start_ts.lt(t.start_ts)].tail(1)
    context.append({'program': f'{t.program_name} ({t.network_code})', 'date': t.broadcast_date.strftime('%Y-%m-%d'),
        'episode': t.episode_name, 'start': t.start_ts.strftime('%H:%M'), 'repeat': t.is_repeat,
        'lead-in': prior.program_name.iloc[0] if len(prior) else '',
        'lead-in episode': prior.episode_name.iloc[0] if len(prior) else ''})
display(pd.DataFrame(context))
heavy_misses = forecast.nlargest(5, 'Heavy buyers APE').copy()
heavy_misses['date'] = heavy_misses.broadcast_date.dt.strftime('%Y-%m-%d')
display(heavy_misses[['program_name', 'network_code', 'date',
    'Heavy buyers rating forecast', 'Heavy buyers rating', 'Heavy buyers APE']].round(3))
# Full evening context exposes sporting overruns and displaced slots.
for net, day in misses[['network_code', 'broadcast_date']].drop_duplicates().itertuples(index=False):
    evening = programs.loc[programs.network_code.eq(net) & programs.broadcast_date.eq(day)
        & programs.start_ts.dt.hour.ge(16) & programs.start_ts.dt.hour.le(23)].copy()
    evening['start'] = evening.start_ts.dt.strftime('%H:%M')
    evening['end'] = evening.end_ts.dt.strftime('%H:%M')
    print(net, day.strftime('%Y-%m-%d'))
    display(evening[['program_name', 'episode_name', 'start', 'end', 'is_repeat']])

,program,date,forecast,actual,P2+ APE,Heavy buyers APE
80,Krapopolis (FOX),2024-10-27,1.062,0.218,3.878,2.534
78,Universal Basic Guys (FOX),2024-10-27,0.676,0.258,1.622,3.926
37,Fire Country (CBS),2024-10-27,1.698,0.666,1.548,2.377
65,Accused (FOX),2024-10-22,0.950,0.373,1.546,1.452
4,What Would You Do? (ABC),2024-10-23,0.438,0.184,1.381,2.024


,program,date,episode,start,repeat,lead-in,lead-in episode
0,Krapopolis (FOX),2024-10-27,Mr. Boogens,21:30,False,Bob's Burgers,For Whom the Doll Toes
1,Universal Basic Guys (FOX),2024-10-27,Fight or Flight,20:30,False,The Simpsons,Shoddy Heat
2,Fire Country (CBS),2024-10-27,NaN,22:44,True,The Equalizer,Haunted Heights
3,Accused (FOX),2024-10-22,Marcus' Story,21:00,False,Murder in a Small Town,A Touch of Panic
4,What Would You Do? (ABC),2024-10-23,Episode 11,22:00,False,Abbott Elementary,Class Pet


,program_name,network_code,date,Heavy buyers rating forecast,Heavy buyers rating,Heavy buyers APE
78,Universal Basic Guys,FOX,2024-10-27,0.953,0.193,3.926
80,Krapopolis,FOX,2024-10-27,1.248,0.353,2.534
37,Fire Country,CBS,2024-10-27,2.066,0.612,2.377
4,What Would You Do?,ABC,2024-10-23,0.345,0.114,2.024
21,The Summit,CBS,2024-10-23,0.772,0.304,1.541


FOX 2024-10-27


,program_name,episode_name,start,end,is_repeat
3063,NFL Football,NFL Football: regional coverage (2 games),16:05,19:15,False
3064,Local / Syndicated Programming,NaN,19:15,20:00,False
3065,The Simpsons,Shoddy Heat,20:00,20:30,False
3066,Universal Basic Guys,Fight or Flight,20:30,21:00,False
3067,Bob's Burgers,For Whom the Doll Toes,21:00,21:30,False
3068,Krapopolis,Mr. Boogens,21:30,22:00,False
3069,Bob's Burgers,NaN,22:00,22:30,True
3070,Krapopolis,NaN,22:30,23:00,True
3071,Local / Syndicated Programming,NaN,23:00,19:00,False


CBS 2024-10-27


,program_name,episode_name,start,end,is_repeat
1537,Local / Syndicated Programming,NaN,16:10,16:25,False
1538,NFL Football,Chicago Bears at Washington Commanders,16:25,19:35,False
1539,60 Minutes,Deportation | Sanctions | Surfmen,19:35,20:00,False
1540,Poppa's House,NaN,20:00,20:30,True
1541,Fire Country,NaN,20:30,20:44,True
1542,Tracker,Bloodlines,20:44,21:44,False
1543,The Equalizer,Haunted Heights,21:44,22:44,False
1544,Fire Country,NaN,22:44,23:00,True
1545,Local / Syndicated Programming,NaN,23:00,07:00,False


FOX 2024-10-22


,program_name,episode_name,start,end,is_repeat
3036,Murder in a Small Town,A Touch of Panic,20:00,21:00,False
3037,Accused,Marcus' Story,21:00,22:00,False
3038,Universal Basic Guys,NaN,22:00,22:30,True
3039,Murder in a Small Town,NaN,22:30,23:00,True
3040,Local / Syndicated Programming,NaN,23:00,20:00,False


ABC 2024-10-23


,program_name,episode_name,start,end,is_repeat
308,ABC World News Tonight with David Muir,Episode 285,18:30,19:00,False
309,Local / Syndicated Programming,NaN,19:00,20:00,False
310,The Golden Bachelorette,Week 6,20:00,21:30,False
311,Abbott Elementary,Class Pet,21:30,22:00,False
312,What Would You Do?,Episode 11,22:00,23:00,False
313,Local / Syndicated Programming,NaN,23:00,00:35,False


**Answer:** Krapopolis (FOX) and Universal Basic Guys (FOX) on 2024-10-27 are the two largest P2+ percentage misses: their actual ratings are only 0.22 and 0.26, so even modest point errors become large percentages. FOX's prior Sunday had KC–SF football until 19:35 and Universal Basic Guys at 20:00; on October 27 football ended at 19:15, followed by local programming, The Simpsons at 20:00 and Universal Basic Guys at 20:30, so the history average mixes different lead-ins and slots. Fire Country (CBS) on October 27 is a repeat fragment at 22:44–23:00 after The Equalizer, on an evening when football ends at 19:35 and Tracker and The Equalizer start at 20:44 and 21:44; it is not comparable to a normal full episode. Accused (FOX), Marcus' Story, aired October 22 at 21:00 after Murder in a Small Town, while What Would You Do? (ABC), Episode 11, aired October 23 at 22:00 after Abbott Elementary: the schedule shows regular slots, so no specific disruption can be established for those two misses. Content variation, competition and panel noise are plausible explanations, not demonstrated causes. Heavy buyers have fewer contributing panelists and smaller actual ratings, magnifying sampling variability and percentage errors: their MAPE is 38.6% versus 35.8% for P2+. A better forecast would use episode/repeat status and the actual slot, duration, lead-in and competing sports/special schedule, with shrinkage for sparse target cells. MAPE is a backtest average, not a confidence interval or the probability that a guarantee will fail.

## Task 5.1 — The plan

In [11]:
# An illustrative one-week buy: at most two spots per observed average weekly
# telecast, rounded up, per program. This keeps cheap cable inventory finite.
# Exact integer search maximizes projected heavy-buyer impressions under that cap.
from scipy.optimize import milp, Bounds, LinearConstraint
costs = program_table['cost per :30'].to_numpy()
target_imps = program_table['target viewers (000)'].to_numpy()*1000
caps = np.ceil(program_table.telecasts.to_numpy()/31*7).astype(int)*2
solution = milp(c=-target_imps/1e6, integrality=np.ones(len(costs)),
                bounds=Bounds(np.zeros(len(costs)), caps),
                constraints=LinearConstraint(costs[None, :], -np.inf, 1_000_000),
                options={'mip_rel_gap': 0})
assert solution.success
spots = np.rint(solution.x).astype(int)
plan = program_table.loc[spots > 0, ['cost per :30', 'target viewers (000)', 'composition', 'target CPM', 'index', 'adult viewers']].copy()
plan['spots'] = spots[spots > 0]
plan['cost'] = plan.spots*plan['cost per :30']
plan['target impressions'] = plan.spots*plan['target viewers (000)']*1000
plan_cost = plan.cost.sum()
plan_imps = plan['target impressions'].sum()
adult_imps = (plan.spots*plan['adult viewers']).sum()
spill_share = 1-plan_imps/adult_imps
spill_cost = (plan.cost*(1-plan.composition)).sum()
assert plan_cost <= 1_000_000 and (plan.spots > 0).all()
display(plan[['spots', 'cost', 'target impressions', 'composition', 'target CPM', 'index']].style.format({
    'spots': '{:.0f}', 'cost': '{:.0f}', 'target impressions': '{:.0f}',
    'composition': '{:.3f}', 'target CPM': '{:.0f}', 'index': '{:.1f}'}))
print('Plan cost ($):', f'{plan_cost:.0f}')
print('Plan target impressions (M):', f'{plan_imps/1e6:.1f}')
print('Unspent budget ($):', f'{1_000_000-plan_cost:.0f}')
print('Non-target share of adult impressions:', f'{spill_share:.3f}')
print('Composition-allocated non-target spend ($):', f'{spill_cost:.0f}')
print('Plan target CPM ($):', f'{plan_cost/plan_imps*1000:.0f}')

,spots,cost,target impressions,composition,target CPM,index
program,,,,,,
Dancing with the Stars (ABC),3,475800,4692694,0.191,101,96.8
Hannity (FNC),8,319200,3335074,0.152,96,77.1
Unsellable Houses (HGTV),6,77400,1903480,0.249,41,126.8
Halloween Wars (FOOD),6,58200,971557,0.228,60,116.2
Grotesquerie (FX),9,68328,1118120,0.188,61,95.9


Plan cost ($): 998928
Plan target impressions (M): 12.0
Unspent budget ($): 1072
Non-target share of adult impressions: 0.814
Composition-allocated non-target spend ($): 814491
Plan target CPM ($): 83


## Task 5.2 — Recommendation

**Answer:** Buy 3 :30 spots in Dancing with the Stars (ABC), 8 in Hannity (FNC), 6 in Unsellable Houses (HGTV), 6 in Halloween Wars (FOOD) and 9 in Grotesquerie (FX), costing $998,928 and leaving $1,072 unspent. At October average delivery this buys 12.0 million gross heavy-buyer impressions at an $83 target CPM, with repeated viewers counted each time. The finite-inventory optimization prioritizes efficient HGTV, FOOD and FX spots, then adds Hannity and Dancing with the Stars for volume; Hannity's 77.1 index illustrates why affinity alone is not the buying rule. These are illustrative November availability assumptions of at most two spots per rounded-up average weekly October telecast, not confirmed inventory, and the seller must confirm that the programs (especially seasonal Halloween Wars) or comparable replacements actually air. Heavy-buyer MAPE of 38.6% versus 35.8% for P2+ warns against treating the impression forecast as a guarantee, particularly since this backtest covers big networks rather than validating the cable shows. About 81.4% of adult impressions go to non-heavy buyers, and allocating each program's cost by its composition assigns about $814,491 to that spill; this is an accounting approximation within adults, excluding children, rather than a separately priced wasted purchase. After the week, measure actual target average-minute impressions, reach, frequency, composition and CPM against the booked guarantee, reconcile any shortfall for make-goods, and use purchase-linked lift measurement to check whether exposure changes buying.

## Bonus

In [12]:
print('Kaggle username:', 'Not submitted (optional competition)')

Kaggle username: Not submitted (optional competition)


**Answer:** The optional competition was not entered. No Kaggle identity, private leaderboard score or submission is claimed.

In [13]:
assert check['your count'].eq(check.SAMPLE).all()
assert len(program_table) == 8
assert np.allclose(weight_sums, 125_800_000)
assert ratings['Heavy buyers viewers'].le(ratings['Adults 18+ viewers'] + 1e-8).all()
assert ratings['Adults 18+ viewers'].le(ratings['P2+ viewers'] + 1e-8).all()
assert np.isfinite(program_table['target CPM']).all()
print('All consistency checks passed.')

All consistency checks passed.
